In [0]:
-- 30_gold_ohlc_features.sql
-- Build Gold mart: OHLC 1m + basic features from Silver
-- Source: enterprise.silver_market.crypto_ohlc_1m
-- Target: enterprise.gold_market.ohlc_1m

CREATE OR REPLACE TABLE enterprise.gold_market.ohlc_1m
USING DELTA
PARTITIONED BY (p_date)
AS
WITH base AS (
  SELECT
    source,
    symbol,
    bar_start_ts,
    bar_end_ts,
    open,
    high,
    low,
    close,
    volume,
    p_date,
    ingestion_ts
  FROM enterprise.silver_market.crypto_ohlc_1m
),
feat AS (
  SELECT
    *,
    -- previous close within (source, symbol)
    LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts) AS prev_close,

    -- price change features
    (close - open) AS candle_body,
    (high - low)   AS candle_range,

    -- simple returns
    CASE
      WHEN LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts) IS NULL THEN NULL
      WHEN LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts) = 0 THEN NULL
      ELSE (close / LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts) - 1)
    END AS ret_1m,

    -- log return
    CASE
      WHEN LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts) IS NULL THEN NULL
      WHEN LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts) <= 0 THEN NULL
      ELSE LN(close / LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts))
    END AS log_ret_1m

  FROM base
),
rolling AS (
  SELECT
    *,
    -- rolling windows: 15m, 60m (within symbol)
    AVG(close) OVER (
      PARTITION BY source, symbol
      ORDER BY bar_start_ts
      ROWS BETWEEN 14 PRECEDING AND CURRENT ROW
    ) AS ma_close_15m,

    AVG(close) OVER (
      PARTITION BY source, symbol
      ORDER BY bar_start_ts
      ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
    ) AS ma_close_60m,

    -- rolling volatility (stddev of log returns)
    STDDEV_POP(log_ret_1m) OVER (
      PARTITION BY source, symbol
      ORDER BY bar_start_ts
      ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
    ) AS vol_logret_60m
  FROM feat
)
SELECT
  source,
  symbol,
  bar_start_ts,
  bar_end_ts,
  open,
  high,
  low,
  close,
  volume,

  -- features
  prev_close,
  candle_body,
  candle_range,
  ret_1m,
  log_ret_1m,
  ma_close_15m,
  ma_close_60m,
  vol_logret_60m,

  -- lineage / audit
  ingestion_ts,
  p_date,
  current_timestamp() AS mart_ts
FROM rolling;


In [0]:
SELECT *
FROM enterprise.gold_market.ohlc_1m
ORDER BY bar_start_ts DESC
LIMIT 50;


In [0]:
SELECT
  symbol,
  p_date,
  COUNT(*) AS bars,
  MIN(bar_start_ts) AS min_ts,
  MAX(bar_start_ts) AS max_ts
FROM enterprise.gold_market.ohlc_1m
GROUP BY symbol, p_date
ORDER BY p_date DESC, symbol;


In [0]:
SELECT 
  symbol,
  bar_start_ts,
  close,
  prev_close,
  ret_1m,
  -- 验证 1: 收益率计算是否正确? (当前收盘 / 上期收盘 - 1)
  ABS(ret_1m - ((close / prev_close) - 1)) AS diff_ret,
  -- 验证 2: 蜡烛实体计算是否正确?
  (close - open) - candle_body AS diff_body
FROM enterprise.gold_market.ohlc_1m
WHERE 
  prev_close IS NOT NULL -- 跳过第一行
  AND (
    -- 允许极小的浮点数误差 (Floating point epsilon)
    ABS(ret_1m - ((close / prev_close) - 1)) > 0.00000001
    OR 
    ABS((close - open) - candle_body) > 0.00000001
  )
ORDER BY bar_start_ts DESC;

In [0]:
WITH time_diff AS (
  SELECT 
    symbol,
    bar_start_ts,
    -- 计算当前行与上一行的时间差（秒）
    unix_timestamp(bar_start_ts) - unix_timestamp(LAG(bar_start_ts) OVER (PARTITION BY symbol ORDER BY bar_start_ts)) AS diff_seconds
  FROM enterprise.gold_market.ohlc_1m
  WHERE p_date >= '2026-01-25' -- 只验证正确日期的范围
)
SELECT * FROM time_diff 
WHERE diff_seconds != 60 -- 找出间隔不是 60 秒的地方
AND diff_seconds IS NOT NULL; -- 忽略每天的第一条

In [0]:
SELECT 
  source, 
  symbol, 
  bar_start_ts, 
  open, 
  high, 
  low, 
  close
FROM enterprise.gold_market.ohlc_1m
WHERE 
  p_date >= '2026-01-25' -- 只查最近的正确数据
  AND (
    high < low          -- 错误：最高价小于最低价（绝对不可能）
    OR high < open      -- 错误：最高价小于开盘价
    OR high < close     -- 错误：最高价小于收盘价
    OR low > open       -- 错误：最低价大于开盘价
    OR low > close      -- 错误：最低价大于收盘价
  );

In [0]:
SELECT * FROM enterprise.gold_market.ohlc_1m
WHERE symbol = 'ETHUSDT' 
  AND p_date = '2026-01-27' 
  AND bar_start_ts = '2026-01-27T12:00:00+00:00'; -- 选一个整点时间